In [ ]:
%pip install pandas matplotlib seaborn scikit-learn xgboost

: 

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data (../data/ used because notebook is inside the notebooks/ folder)
train_df = pd.read_csv('../data/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('../data/UNSW_NB15_testing-set.csv')

print("Training Data Shape:", train_df.shape)
print("Testing Data Shape:", test_df.shape)

# Look at the first 5 rows
train_df.head()

Training Data Shape: (82332, 45)
Testing Data Shape: (175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve
import numpy as np

# ১. অপ্রয়োজনীয় কলাম বাদ দেওয়া (id এবং attack_cat দরকার নেই বাইনারি ক্লাসিফিকেশনের জন্য)
train_df = train_df.drop(['id', 'attack_cat'], axis=1)
test_df = test_df.drop(['id', 'attack_cat'], axis=1)

# ২. Categorical encoding (লেখাগুলোকে নম্বরে রূপান্তর)
cols_to_encode = ['proto', 'service', 'state']
le = LabelEncoder()
for col in cols_to_encode:
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.fit_transform(test_df[col])

# ৩. Train-Test Split (ফিচার এবং লেবেল আলাদা করা)
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']
X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

# ৪. Scaling (সব ভ্যালুকে একই স্কেলে আনা)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ৫. Baseline Model: Random Forest
rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

# ৬. Threshold Calibration (FPR < 3% বের করা)
probs = rf_model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, probs)

# ৩% FPR এর কাছাকাছি থ্রেশহোল্ড খুঁজে বের করা
idx = np.where(fpr <= 0.03)[0][-1]
optimal_threshold = thresholds[idx]

print(f"Optimal Threshold for FPR < 3%: {optimal_threshold:.4f}")
print(f"At this threshold, Detection Rate (TPR) is: {tpr[idx]*100:.2f}%")

Optimal Threshold for FPR < 3%: 0.4771
At this threshold, Detection Rate (TPR) is: 87.77%
